# HLS Foundation Model Finetuning notebook

This notebook demonstrates the steps to fintune the HLS foundation model (A.K.A Prithvi) which is trained using HLSL30 and HLSS30 datasets. 

Note: Entierty of this notebook is desigend to work well within the AWS sagemaker environment. AWS sagemaker environment access for your account can be found using https://creds-workshop.nasa-impact.net/

![HLS Training](../images/HLS-training.png)

In [ ]:
!pwd

/home/oepeng_aims_ac_za/Code/prithvi-global-workshop/notebooks


In [2]:
# Install required packages
# !pip install -r ../requirements.txt

# Create directories needed for data, model, and config preparations
!mkdir datasets
# !mkdir models
# !mkdir configs

## Dataset preparation

For this hands-on session, Burn Scars example will be used for fine-tuning. All of the data and pre-trained models are available in Huggingface. Huggingface packages and git will be utilized to download, and prepare datasets and pretrained models.


### Download HLS Burn Scars dataset from Huggingface: https://huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars

In [4]:
!git clone https://huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars datasets

Cloning into 'datasets'...
remote: Enumerating objects: 1724, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (21/21), done.


remote: Total 1724 (delta 34), reused 55 (delta 34), pack-reused 1669 (from 1)
Receiving objects: 100% (1724/1724), 259.96 KiB | 11.82 MiB/s, done.
Resolving deltas: 100% (63/63), done.


In [2]:
!git lfs --version

git-lfs/3.5.1 (GitHub; linux amd64; go 1.21.11)


In [1]:
!du -sh datasets/*

4.0K	datasets/hls_burn_scars.py
2.5G	datasets/hls_burn_scars.tar.gz
4.0K	datasets/README.md
3.5G	datasets/training
1.7G	datasets/validation


In [9]:
!tar -xzf datasets/hls_burn_scars.tar.gz -C datasets/

## Download config and Pre-trained model

The HLS Foundation Model (pre-trained model), and configuration for Burn Scars downstream task are available in Huggingface. We use `huggingface_hub` python package to download the files locally.

In [8]:
!gsutil ls

gs://chum-bkt/


In [2]:
# Define constants
BUCKET_NAME = 'gs://chum-bkt' # Replace this with the bucket name available from http://smd-ai-workshop-creds-webapp.s3-website-us-east-1.amazonaws.com/ 
CONFIG_PATH = './configs'
DATASET_PATH = './datasets'
MODEL_PATH = './models'

In [10]:
# Download pre-trained model file from huggingface
!curl -o models/prithvi_global_v1.pt https://www.nsstc.uah.edu/data/sujit.roy/Prithvi_checkpoints/checkpoint.pt 

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 3786M  100 3786M    0     0  58.1M      0  0:01:05  0:01:05 --:--:-- 56.5M2M      0  0:01:05  0:00:35  0:00:30 55.7M


In [ ]:
# # Prepare sagemaker session with files uploaded to s3 bucket
# import sagemaker

# sagemaker_session = sagemaker.Session()
# train_images = sagemaker_session.upload_data(path='datasets/training', bucket=BUCKET_NAME, key_prefix='data/training')
# val_images = sagemaker_session.upload_data(path='datasets/validation', bucket=BUCKET_NAME, key_prefix='data/validation')
# test_images = sagemaker_session.upload_data(path='datasets/validation', bucket=BUCKET_NAME, key_prefix='data/test')




In [3]:
!gsutil -m cp -r datasets/{training,validation} gs://chum-bkt/data

Copying file://datasets/training/subsetted_512x512_HLS.S30.T14SNB.2018215.v1.4_merged.tif [Content-Type=image/tiff]...
Copying file://datasets/training/subsetted_512x512_HLS.S30.T16RFT.2019250.v1.4_merged.tif [Content-Type=image/tiff]...
Copying file://datasets/training/subsetted_512x512_HLS.S30.T11SKV.2019152.v1.4_merged.tif [Content-Type=image/tiff]...
Copying file://datasets/training/subsetted_512x512_HLS.S30.T11TMG.2018222.v1.4_merged.tif [Content-Type=image/tiff]...
Copying file://datasets/training/subsetted_512x512_HLS.S30.T13TCL.2020245.v1.4_merged.tif [Content-Type=image/tiff]...
Copying file://datasets/training/subsetted_512x512_HLS.S30.T13TDM.2020250.v1.4_merged.tif [Content-Type=image/tiff]...
Copying file://datasets/training/subsetted_512x512_HLS.S30.T16SEB.2019135.v1.4_merged.tif [Content-Type=image/tiff]...
Copying file://datasets/training/subsetted_512x512_HLS.S30.T15SWA.2021093.v1.4_merged.tif [Content-Type=image/tiff]...
Copying file://datasets/training/subsetted_512x5

In [7]:
# Rename configuration file name to user specific filename
import os

identifier = 'workshop-015' # Please update this with an identifier

config_filename = '../configs/burn_scars.yaml'
new_config_filename = f"../configs/{identifier}-burn_scars.yaml"
os.rename(config_filename, new_config_filename)

FileNotFoundError: [Errno 2] No such file or directory: 'configs/burn_scars.yaml' -> 'configs/workshop-015-burn_scars.yaml'

In [6]:
from pathlib import Path

identifier = 'workshop-015'  # Please update this with an identifier

config_filename = Path('../configs/burn_scars.yaml')
new_config_filename = config_filename.with_name(f"{identifier}-burn_scars.yaml")

config_filename.rename(new_config_filename)


PosixPath('../configs/workshop-015-burn_scars.yaml')

In [ ]:
# Upload config files to s3 bucket
configs = sagemaker_session.upload_data(path=new_config_filename, bucket=BUCKET_NAME, key_prefix='data/configs')
new_config_filename = new_config_filename.replace('../', '') # point to cwd

In [9]:
!gsutil cp -r ../configs/workshop-* gs://chum-bkt/data/configs

Copying file://../configs/workshop-015-burn_scars.yaml [Content-Type=application/octet-stream]...
/ [1 files][  1.4 KiB/  1.4 KiB]                                                
Operation completed over 1 objects/1.4 KiB.                                      


In [ ]:
models = sagemaker_session.upload_data(path='models/prithvi_global_v1.pt', bucket=BUCKET_NAME, key_prefix='data/models')

In [11]:
!gsutil -m cp -r models/*.pt gs://chum-bkt/data/models

Copying file://models/prithvi_global_v1.pt [Content-Type=application/vnd.snesdev-page-table]...
==> NOTE: You are uploading one or more large file(s), which would run          
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

Resuming upload for file://models/prithvi_global_v1.pt
- [1/1 files][  3.7 GiB/  3.7 GiB] 100% Done 118.6 MiB/s ETA 00:00:00           
Operation completed over 1 objects/3.7 GiB.                                      



Note: For HLS Foundation Model, MMCV and MMSEG were used. These libraries use pytorch underneath them for training, data distribution etc. However, these packages are not available in sagemaker by default. Thus, custom script training is required. Sagemaker utilizes Docker for custom training scripts. If interested, the code included in the image we are using for training (574347909231.dkr.ecr.us-west-2.amazonaws.com/prithvi_global:latest) is bundled with this repository, and the train script used is `train.py`.

The current HLS Foundation model fits in a single NVIDIA Tesla V100 GPU (16GB VRAM). Hence, `ml.p3.2xlarge` instance is used for training.

In [ ]:
# # Setup variables for training using sagemaker
# from datetime import time
# from sagemaker import get_execution_role
# from sagemaker.estimator import Estimator


# name = f'{identifier}-sagemaker'
# role = get_execution_role()
# input_s3_uri = f"s3://{BUCKET_NAME}/data"
# finetuned_model_name = f"{identifier}-workshop.pth"
# environment_variables = {
#     'CONFIG_FILE': f"/opt/ml/data/configs/{new_config_filename.split('/')[-1]}",
#     'MODEL_DIR': "/opt/ml/data/models/",
#     'MODEL_NAME': finetuned_model_name,
#     'S3_URL': input_s3_uri,
#     'BUCKET_NAME': BUCKET_NAME,
#     'ROLE_ARN': role,
#     'ROLE_NAME': role.split('/')[-1],
#     'EVENT_TYPE': 'burn_scars',
#     'VERSION': 'v1'
# }

# ecr_container_url = '574347909231.dkr.ecr.us-west-2.amazonaws.com/prithvi_global:latest'
# sagemaker_role = 'SageMaker-ExecutionRole-20240206T151814'

# instance_type = 'ml.p3.2xlarge'

# instance_count = 1
# memory_volume = 50

In [8]:
# Setup variables for training using GCP's Vertex AI
from datetime import time
from google.cloud import aiplatform

# Replace with your GCP project ID and region
PROJECT_ID = "my-colab-99752"
REGION = "us-central1"

# Variables for GCP
finetuned_model_name = f"{identifier}-workshop.pth"
input_gcs_uri = f"gs://{BUCKET_NAME}/data"  # GCS URI for training data


# Set up environment variables for GCP's Vertex AI training job
environment_variables = {
    'CONFIG_FILE': f"/opt/ml/data/configs/{new_config_filename.name}",
    'MODEL_DIR': "/opt/ml/data/models/",
    'MODEL_NAME': finetuned_model_name,
    'GCS_URL': input_gcs_uri,
    'BUCKET_NAME': BUCKET_NAME,
    'EVENT_TYPE': 'burn_scars',
    'VERSION': 'v1'
}


In [12]:
BUCKET_NAME

'gs://chum-bkt'

In [17]:
# Define the Vertex AI training parameters

# Model name for the container image in Google Container Registry (GCR)
ecr_container_url = f'gcr.io/<{PROJECT_ID}/prithvi_global:latest'

# Instance configuration for GCP
instance_type = 'n1-standard-4'  # For example, adjust based on your needs
instance_count = 1
memory_volume = 50  # Adjust according to your requirements


# Initialize the Vertex AI training job

aiplatform.init(project=PROJECT_ID, location=REGION,staging_bucket=BUCKET_NAME )
from google.cloud import aiplatform

# Now, create the custom job
from google.cloud import aiplatform

# Define the CustomJob using a Docker container
custom_job = aiplatform.CustomJob(
    display_name="your-training-job-name",
    worker_pool_specs=[
        {
            "machine_spec": {
                "machine_type": "n1-standard-4",  # Choose the machine type you want
                "accelerator_type": "NVIDIA_TESLA_T4",  # Specify GPU if needed
                "accelerator_count": 1,  # Adjust as needed
            },
            "replica_count": 1,
            "disk_spec": {
                "boot_disk_type": "pd-standard", 
                "boot_disk_size_gb": 100,  # Specify disk size for training
            },
            "container_spec": {
                "image_uri": "gcr.io/your-project-id/your-image-name:tag",  # Use your Docker image
            },
        }
    ],
)

# Run the training job
custom_job.run(sync=True)


# Run the training job
custom_job.run(sync=True)

Creating CustomJob


PermissionDenied: 403 Permission 'aiplatform.customJobs.create' denied on resource '//aiplatform.googleapis.com/projects/my-colab-99752/locations/us-central1' (or it may not exist). [reason: "IAM_PERMISSION_DENIED"
domain: "aiplatform.googleapis.com"
metadata {
  key: "resource"
  value: "projects/my-colab-99752/locations/us-central1"
}
metadata {
  key: "permission"
  value: "aiplatform.customJobs.create"
}
]

In [ ]:
# Establish an estimator (model) using sagemaker and the configurations from the previous cell.
estimator = Estimator(image_uri=ecr_container_url,
                      role=get_execution_role(),
                      base_job_name=name,
                      instance_count=1,
                      environment=environment_variables,
                      instance_type=instance_type)


In [ ]:
# Start training
estimator.fit()

## Deploy trained model to sagemaker endpoint

In [ ]:
# Since we are downloading data from the internet, platform is used. Else, VPC is prefered.
image_config = {
     'RepositoryAccessMode': 'Platform'
}

In [ ]:
IMAGE_URI = '574347909231.dkr.ecr.us-west-2.amazonaws.com/prithvi_global_inference'

ENV = {
    "CHECKPOINT_FILENAME": f"s3://{BUCKET_NAME}/models/{finetuned_model_name}",
    "S3_CONFIG_FILENAME": f"s3://{BUCKET_NAME}/data/{new_config_filename}",
    "BUCKET_NAME": BUCKET_NAME,
    "AIP_PREDICT_ROUTE": "/invocations",
    "BACKBONE_FILENAME": f"s3://{BUCKET_NAME}/data/models/prithvi_global_v1.pt"
}

primary_container = {
    'ContainerHostname': 'ModelContainer',
    'Image': IMAGE_URI,
    'ImageConfig': image_config,
    'Environment': ENV
}

In [ ]:
model_name = f'prithvi-global-{identifier}'
execution_role_arn = get_execution_role()

In [ ]:
import boto3
sagem = boto3.client('sagemaker')

# Create model based on custom code and artifacts in sagemaker
resp = sagem.create_model(
        ModelName=model_name,
        PrimaryContainer=primary_container,
        ExecutionRoleArn=execution_role_arn
    )

endpoint_config_name = f'{model_name}-endpoint-config'

# Create endpoint config for easier deployment
sagem.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            'VariantName': 'v1',
            'ModelName': model_name,
            'InitialInstanceCount': 1,
            'InstanceType': 'ml.p3.2xlarge'
        },
    ],
)

endpoint_name = f'{model_name}-endpoint'

# Create endpoint i.e Deployment.
sagem.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

sagem.describe_endpoint(EndpointName=endpoint_name)

## Delete all resources after testing is done and the resources are no longer needed.

In [ ]:

sagem.delete_model(ModelName=model_name)
sagem.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sagem.delete_endpoint(EndpointName=endpoint_name,)

**Vertex AI Workbench instances are Jupyter notebook-based development environments for the entire data science workflow.**

console >>  Vertex AI Workbench Instances page.